In [ ]:
from ultralytics.data.annotator import auto_annotate

auto_annotate(
    data=r"C:\AI\sampah_sungai\auto_sam",
    det_model=r"C:\AI\sampah_sungai\best.pt",
    sam_model=r"C:\AI\sampah_sungai\Base_model\sam_l.pt",
    device="cuda:0",         # Ganti 'cpu' kalau kamu tidak pakai GPU
    conf=0.5,                # Threshold confidence — bisa kamu sesuaikan
    iou=0.5,                 # IoU NMS
    imgsz=640,               # Ukuran input, sesuaikan dengan training-mu
    max_det=50,             # Jumlah maksimum objek per gambar
    output_dir=r"C:\AI\sampah_sungai\auto_sam\label"  # Output folder label segmentasi
)


image 1/23 C:\AI\sampah_sungai\auto_sam\000000020_png_jpg.rf.d090363237b4f2399f610fff65502b49.jpg: 640x640 2 sampahs, 45.4ms


In [ ]:
#-----------------------------------------------------------#
############# STEP 02 : Check hasil Augmentasi ##############
#-----------------------------------------------------------#

import cv2              # Untuk manipulasi gambar
import numpy as np      # Untuk operasi array dan numerik
import random           # Untuk pengacakan (misal: memilih gambar acak, warna)
import os               # Untuk operasi file dan folder
import glob             # Untuk pencarian file dengan pola tertentu
import logging
from tqdm import tqdm   # Untuk progress bar di terminal
import shutil

# ============================================
# Definisi Folder Sumber dan Tujuan
# ============================================
aug_img_dir = r"C:\AI\sampah_sungai\auto_sam"
aug_label_dir = r"C:\AI\sampah_sungai\auto_sam\label"
save_dir = r"C:\AI\sampah_sungai\auto_sam\Cek"

# ============================================
# Konfigurasi Logging
# ============================================
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()  # Hapus handler default

# Handler untuk menyimpan log ke file (INFO ke atas)
file_handler = logging.FileHandler("proses.log")
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Handler untuk menampilkan log ke terminal (WARNING ke atas)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_formatter = logging.Formatter('%(levelname)s: %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

# ============================================
# Pastikan Folder Tujuan Ada dan Kosong
# ============================================
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
    logger.info("Folder '%s' dibuat.", save_dir)
else:
    # Kosongkan folder cek jika sudah ada
    for filename in os.listdir(save_dir):
        file_path = os.path.join(save_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            logger.warning("Gagal menghapus %s. Error: %s", file_path, e)
    logger.info("Folder '%s' dikosongkan.", save_dir)

# ============================================
# Pengambilan File Gambar
# ============================================
img_files = glob.glob(os.path.join(aug_img_dir, "*.jpg"))

if len(img_files) < 1000000:
    logger.warning("Gambar kurang dari 100! Menampilkan semua yang ada.")
    random_imgs = img_files
else:
    random_imgs = random.sample(img_files, 100)

logger.info("Menampilkan %d gambar untuk pengecekan.", len(random_imgs))

# ============================================
# Fungsi Bantuan
# ============================================

def compute_polygon_area(pts):
    """Menghitung luas poligon menggunakan metode contourArea OpenCV."""
    return cv2.contourArea(pts)

def is_polygon_within_bounds(pts, width, height):
    """Memeriksa apakah semua titik poligon berada dalam batas gambar."""
    for pt in pts.reshape(-1, 2):
        x, y = pt
        if x < 0 or x > width or y < 0 or y > height:
            return False
    return True

def round_polygon(pts, precision=4):
    """Mengembalikan tuple dari koordinat poligon yang sudah dibulatkan."""
    return tuple(np.round(pts.flatten(), precision))

# ============================================
# Proses Pengolahan dan Pengecekan Tiap Gambar
# ============================================
for img_file in tqdm(random_imgs, desc="Memproses gambar"):
    base_name = os.path.splitext(os.path.basename(img_file))[0]
    label_file = os.path.join(aug_label_dir, base_name + ".txt")
    
    if not os.path.exists(label_file):
        logger.warning("Label untuk %s tidak ditemukan, lewati.", base_name)
        continue
    
    image = cv2.imread(img_file)
    if image is None:
        logger.warning("Gagal membaca %s, lewati.", img_file)
        continue
    
    h, w, _ = image.shape
    original_image = image.copy()  # Untuk menggambar anotasi
    
    with open(label_file, "r") as f:
        lines = f.readlines()
    
    # Dictionary untuk mendeteksi duplikasi: key = (class_id, rounded koordinat)
    seen_polygons = {}
    duplicate_found = False
    
    for idx, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 3:
            logger.warning("Format label salah pada %s baris %d.", base_name, idx+1)
            continue

        # Parsing label dan koordinat
        class_id = parts[0]
        try:
            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        except Exception as e:
            logger.warning("Gagal parsing koordinat pada %s baris %d. Error: %s", base_name, idx+1, e)
            continue
        
        # Konversi koordinat relatif ke piksel
        coords[:, 0] *= w
        coords[:, 1] *= h
        coords = coords.astype(np.int32)
        pts = coords.reshape((-1, 1, 2))
        
        # Periksa apakah poligon berada dalam batas gambar
        if not is_polygon_within_bounds(pts, w, h):
            logger.warning("Poligon pada %s baris %d berada di luar batas gambar.", base_name, idx+1)
            # Tandai dengan warna oranye
            color = (0, 165, 255)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=3)
        else:
            # Tandai dengan warna hijau jika valid
            color = (0, 255, 0)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=2)
        
        # Periksa area poligon
        area = compute_polygon_area(pts)
        if area < 10:
            logger.warning("Area poligon terlalu kecil (%.2f) pada %s baris %d.", area, base_name, idx+1)
            # Tandai dengan warna biru
            cv2.polylines(image, [pts], isClosed=True, color=(255, 0, 0), thickness=3)
        
        # Cek duplikasi label
        key = (class_id, round_polygon(pts, precision=2))
        if key in seen_polygons:
            logger.warning("Duplikasi label ditemukan pada %s baris %d. Duplikat dengan baris %d.", 
                           base_name, idx+1, seen_polygons[key])
            duplicate_found = True
            # Tandai duplikasi dengan warna merah
            cv2.polylines(image, [pts], isClosed=True, color=(0, 0, 255), thickness=3)
        else:
            seen_polygons[key] = idx+1  # Simpan nomor baris label
        
    # Simpan gambar hasil pengecekan
    save_path = os.path.join(save_dir, base_name + "_checked.jpg")
    cv2.imwrite(save_path, image)
    
    logger.info("Gambar %s telah dicek dan disimpan di %s", base_name + "_checked.jpg", save_dir)
    
    # Jika ditemukan duplikasi, juga simpan gambar asli untuk referensi
    if duplicate_found:
        dup_save_path = os.path.join(save_dir, base_name + "_duplicate.jpg")
        cv2.imwrite(dup_save_path, original_image)
        logger.info("Gambar asli %s juga disimpan sebagai referensi duplikasi.", base_name)

logger.warning("Proses pengecekan selesai!")

In [ ]:
import os
import cv2
import numpy as np
from shapely.geometry import Polygon
from tqdm import tqdm

# Folder input/output
input_folder = r"C:\AI\makanan\Dataset_workshop\bengkel\tempe_label"
output_folder = r"C:\AI\makanan\Dataset_workshop\bengkel\label_simplified"
os.makedirs(output_folder, exist_ok=True)

# Threshold minimum kemiripan bentuk
IOU_THRESHOLD = 0.98
DUPLICATE_IOU_THRESHOLD = 0.95  # Threshold untuk menghapus duplikat

# Daftar nilai EPSILON yang akan dicoba (dari kecil ke besar)
EPSILON_CANDIDATES = [0.00001, 0.00003, 0.00006, 0.00009, 0.0001, 0.0003, 0.0006, 0.0009]

# Statistik
total_objects = 0
total_points_original = 0
total_points_simplified = 0
total_duplicates_removed = 0

def calculate_iou(poly1_coords, poly2_coords):
    try:
        poly1 = Polygon(np.array(poly1_coords).reshape(-1, 2)).buffer(0)
        poly2 = Polygon(np.array(poly2_coords).reshape(-1, 2)).buffer(0)
        if not poly1.is_valid or not poly2.is_valid:
            return 0
        inter = poly1.intersection(poly2).area
        union = poly1.union(poly2).area
        return inter / union if union != 0 else 0
    except:
        return 0

def is_valid_polygon(coords):
    try:
        poly = Polygon(np.array(coords).reshape(-1, 2)).buffer(0)
        return poly.is_valid and poly.area > 0
    except:
        return False

def auto_simplify(coords):
    points = np.array(coords, dtype=np.float32).reshape(-1, 2)

    if len(points) <= 4:
        return coords

    best_result = points
    for eps in EPSILON_CANDIDATES:
        approx = cv2.approxPolyDP(points.reshape((-1, 1, 2)), eps, True)
        simplified = approx.reshape(-1, 2)
        iou = calculate_iou(points, simplified)

        if iou >= IOU_THRESHOLD and is_valid_polygon(simplified):
            best_result = simplified
        else:
            break

    return best_result.reshape(-1).tolist()

# Proses file dengan tqdm
label_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

for file_name in tqdm(label_files, desc="🔄 Memproses label"):
    input_path = os.path.join(input_folder, file_name)
    output_path = os.path.join(output_folder, file_name)

    with open(input_path, "r") as infile:
        lines = infile.readlines()

    simplified_annotations = []

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 3:
            continue

        class_id = 1  # override semua ke class 1
        coords = list(map(float, parts[1:]))
        simplified_coords = auto_simplify(coords)

        if len(simplified_coords) < 6 or not is_valid_polygon(np.array(simplified_coords).reshape(-1, 2)):
            simplified_coords = coords

        simplified_annotations.append((class_id, simplified_coords))
        total_objects += 1
        total_points_original += len(coords) // 2
        total_points_simplified += len(simplified_coords) // 2

    # Hapus duplikat berdasarkan IoU tinggi
    unique_annotations = []
    for i, (cls_i, coords_i) in enumerate(simplified_annotations):
        is_duplicate = False
        for j, (cls_j, coords_j) in enumerate(unique_annotations):
            if cls_i == cls_j:
                iou = calculate_iou(coords_i, coords_j)
                if iou >= DUPLICATE_IOU_THRESHOLD:
                    is_duplicate = True
                    total_duplicates_removed += 1
                    break
        if not is_duplicate:
            unique_annotations.append((cls_i, coords_i))

    # Tulis hasil akhir ke file
    with open(output_path, "w") as outfile:
        for class_id, coords in unique_annotations:
            coords_str = ' '.join(map(str, coords))
            outfile.write(f"{class_id} {coords_str}\n")

# Statistik akhir
print("\n📊 Statistik Perbandingan Anotasi Otomatis")
print("---------------------------------------------")
print(f"🗂️  Jumlah file label         : {len(label_files)}")
print(f"🔸 Total objek (poligon)      : {total_objects}")
print(f"🧩 Total titik (original)     : {total_points_original}")
print(f"🧩 Total titik (simplified)   : {total_points_simplified}")
print(f"📉 Pengurangan total titik    : {total_points_original - total_points_simplified} "
      f"({100 * (total_points_original - total_points_simplified) / total_points_original:.2f}%)")
print(f"❌ Total duplikat dihapus     : {total_duplicates_removed}")

In [ ]:
import os
import shutil
import zipfile

# Path awal
label_simplified_folder = r"C:\AI\makanan\Dataset_workshop\bengkel\label_simplified"
output_structure_root = r"C:\AI\makanan\Dataset_workshop\bengkel\dataset_package"
label_output_folder = os.path.join(output_structure_root, "labels", "train")
os.makedirs(label_output_folder, exist_ok=True)

# Salin semua label ke struktur baru
for file in os.listdir(label_simplified_folder):
    if file.endswith(".txt"):
        src = os.path.join(label_simplified_folder, file)
        dst = os.path.join(label_output_folder, file)
        shutil.copy2(src, dst)

# Buat file data.yml
data_yml_path = os.path.join(output_structure_root, "data.yml")
with open(data_yml_path, "w") as f:
    f.write("""names:
  0: bawang
  1: tempe
path: .
train: train.txt
""")

# Buat file train.txt
train_txt_path = os.path.join(output_structure_root, "train.txt")
image_base_path = "data/images/train"

image_entries = []
for file in os.listdir(label_simplified_folder):
    if file.endswith(".txt"):
        base_name = os.path.splitext(file)[0]
        # Mengganti backslash menjadi garis miring
        image_path = os.path.join(image_base_path, base_name + ".jpg").replace("\\", "/")
        image_entries.append(image_path)

with open(train_txt_path, "w") as f:
    for path in image_entries:
        f.write(path + "\n")

# Kompres jadi ZIP
zip_output_path = os.path.join(output_structure_root, "dataset_package.zip")

with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_structure_root):
        for file in files:
            if file == "dataset_package.zip":
                continue  # Jangan masukkan file zip ke dalam dirinya sendiri
            full_path = os.path.join(root, file)
            relative_path = os.path.relpath(full_path, output_structure_root)
            zipf.write(full_path, arcname=relative_path)

print(f"\n✅ Dataset berhasil dikemas ke dalam: {zip_output_path}")

In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from ultralytics import YOLO

# === Path Config ===
sam_ckpt = r"D:\MyProjects\makanan\Base_model\Yolo\sam2.1_hiera_large.pt"
sam_cfg = r"D:\MyProjects\makanan\Base_model\Yolo\sam2.1_hiera_l.yaml"
yolo_ckpt = r"D:\MyProjects\makanan\eksport\yolo11s-obb(v1).pt"
img_dir = r"D:\MyProjects\makanan\tes\auto_sam"
output_dir = os.path.join(img_dir, "labels")
os.makedirs(output_dir, exist_ok=True)

# === Init Models ===
try:
    predictor = SAM2ImagePredictor(build_sam2(sam_cfg, sam_ckpt).cuda())
    yolo_model = YOLO(yolo_ckpt)
except Exception as e:
    print(f"[FATAL] Model loading error: {e}")
    exit(1)

failed_images = []

# === Loop Gambar ===
for img_name in tqdm(os.listdir(img_dir)):
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    try:
        img_path = os.path.join(img_dir, img_name)
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError("Gagal load image")

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            predictor.set_image(image_rgb)
            masks, _, _ = predictor.predict()

        label_lines = []

        for mask in masks:
            # Ambil contour
            contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contours:
                continue

            contour = max(contours, key=cv2.contourArea)
            polygon = contour.squeeze()

            if polygon.ndim != 2 or len(polygon) < 3:
                continue

            # Bounding Box & Crop
            x, y, bw, bh = cv2.boundingRect(polygon)
            crop = image[y:y+bh, x:x+bw]
            if crop.size == 0:
                continue

            # Run YOLO di crop untuk klasifikasi
            results = yolo_model.predict(source=crop, conf=0.25, verbose=False)
            if len(results) == 0 or len(results[0].boxes) == 0:
                continue

            class_id = int(results[0].boxes.cls[0].item())

            # Normalisasi polygon (pakai dimensi asli gambar, bukan crop)
            polygon_norm = [(px / w, py / h) for px, py in polygon]
            line = f"{class_id} " + " ".join([f"{x:.6f} {y:.6f}" for x, y in polygon_norm])
            label_lines.append(line)

        # Simpan label
        if label_lines:
            out_path = os.path.join(output_dir, Path(img_name).stem + ".txt")
            with open(out_path, "w") as f:
                f.write("\n".join(label_lines))
        else:
            print(f"[SKIP] No label generated for {img_name}")
            failed_images.append(img_name)

    except Exception as e:
        print(f"[ERROR] {img_name}: {e}")
        failed_images.append(img_name)

# === Summary ===
print("\n=== Done ===")
print(f"Gambar gagal/skip: {len(failed_images)}")
if failed_images:
    print("Gambar yang gagal:")
    for name in failed_images:
        print(f"- {name}")


In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

# Paths
image_dir = r"D:\MyProjects\makanan\tes\auto_sam"
label_dir = r"D:\MyProjects\makanan\tes"

colors = [(0, 255, 0), (0, 0, 255), (255, 0, 0)]  # Ganti sesuai jumlah kelas

image_paths = glob(os.path.join(image_dir, "*.jpg")) + glob(os.path.join(image_dir, "*.png"))

def draw_annotations(img_path):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    label_path = os.path.join(label_dir, os.path.basename(img_path).replace(".jpg", ".txt").replace(".png", ".txt"))
    if not os.path.exists(label_path):
        print(f"Tidak ada label untuk {img_path}")
        return

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        cls_id = int(parts[0])
        points = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        points[:, 0] *= w
        points[:, 1] *= h
        points = points.astype(np.int32)
        cv2.polylines(img, [points], isClosed=True, color=colors[cls_id % len(colors)], thickness=2)
        cv2.putText(img, f"Class {cls_id}", tuple(points[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, colors[cls_id % len(colors)], 2)

    # Tampilkan di Jupyter
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()

# Contoh: tampilkan 5 gambar pertama
for path in image_paths[:5]:
    draw_annotations(path)

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path
import cv2

# ==== Konfigurasi ====
WEIGHTS_PATH = Path(r"D:\MyProjects\makanan\Hasil\yolo11s-obb(v1)\weights\best.pt")
SOURCE_DIR   = Path(r"D:\MyProjects\makanan\tes\auto_sam")
OUTPUT_DIR   = Path(r"D:\MyProjects\makanan\auto_labels")
CONF_THRESH  = 0.1   # hanya simpan prediksi dengan confidence >= 0.25
IMG_EXTS     = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

# Buat direktori output
LABELS_DIR = OUTPUT_DIR / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

# Load model YOLO
model = YOLO(str(WEIGHTS_PATH))  # Load model dengan cara yang kamu sebutkan

# Fungsi bantu: konversi kotak ke format YOLO
def xyxy2yolo(x1, y1, x2, y2, img_w, img_h):
    x_center = (x1 + x2) / 2.0 / img_w
    y_center = (y1 + y2) / 2.0 / img_h
    width    = (x2 - x1) / img_w
    height   = (y2 - y1) / img_h
    return x_center, y_center, width, height

# Iterasi setiap gambar di SOURCE_DIR
for img_path in SOURCE_DIR.iterdir():
    if img_path.suffix.lower() not in IMG_EXTS:
        continue

    # Baca gambar untuk mendapatkan dimensi
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    # Inference
    results = model(img)  # inference gambar

    # Ambil prediksi dari hasil inference
    # 'results' berupa list, dan deteksi berada di index pertama (0)
    detections = results[0].boxes

    # Periksa apakah ada deteksi
    if detections is None or len(detections) == 0:
        print(f"⚠️ Tidak ada deteksi pada {img_path.name}")
        continue

    # Siapkan file label
    label_file = LABELS_DIR / f"{img_path.stem}.txt"
    with open(label_file, 'w') as f:
        for det in detections:
            cls = int(det.cls)  # class id
            x1, y1, x2, y2 = det.xyxy  # koordinat kotak pembatas
            conf = det.conf  # confidence
            # Konversi ke format YOLO
            xc, yc, ww, hh = xyxy2yolo(x1, y1, x2, y2, w, h)
            f.write(f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f} {conf:.2f}\n")

    print(f"✅ {img_path.name} → {label_file.name}")

print("🎉 Pseudo‑labeling selesai! Labels tersimpan di:", LABELS_DIR)